# Calculate Derivatives and Other Equations

In [1]:
import subprocess
def gpu_available():
    try:
        subprocess.check_output(['nvidia-smi'])
        return True
    except Exception:
        return False
    
import platform
def is_linux():
    try:
        return platform.system() == 'Linux'
    except Exception:
        return False


import sys
def is_jax():
    if len([m for m in sys.modules.keys() if m.startswith('jax')])>0:
        return True
    return False

In [2]:
import sys
import re

pattern = re.compile(r'.*\.$')
matches = [name for name in sys.modules if pattern.match(name)]

print(matches)

[]


In [3]:
gpu=gpu_available()
linux=is_linux()

jax_=is_jax()  # this is used further into the code to aid in masking, wherease the gpu and linux are used in the following imports

if gpu is True and linux is True:
    print('Linux detected')
    print('GPU detected')
    import pandas as pd
    import cudf.pandas
    cudf.pandas.install()
    import cuml
    from cuml.naive_bayes import CategoricalNB
    #import jax
    #import jax.numpy as np
    #import jax.scipy.stats as jstats  ## this should import jax.scipy.special
    #NEEDS MORE ATTENTION
    #jax_=is_jax()

else: 
    print(f"GPU: {gpu}\nLinux: {linux}")
    print(f"CuDF not available. Running Pandas")
    import pandas as pd
    from sklearn.naive_bayes import CategoricalNB
    import numpy as np




import numpy as np
import scipy.stats
from scipy.special import chdtrc  ###----> for jax: https://docs.jax.dev/en/latest/_autosummary/jax.scipy.stats.chi2.sf.html#jax.scipy.stats.chi2.sf


import matplotlib.pyplot as plt

import seaborn as sns

Linux detected
GPU detected


/home/lelandmesford/Projects/Consumer_Habits/.venv/lib/python3.13/site-packages/cudf/pandas/__init__.py:51: UserWarning: Failed to check cudaDevAttrConcurrentManagedAccess with error 35
  warnings.warn(str(e))


In [ ]:
import sympy
from sympy import *
from sympy.abc import a,m,x

In [2]:
x = Symbol('x')
limit(sin(x)/x, x, 0)
integrate(1/x, x)

log(x)

pareto pmf, cdf

In [ ]:
#shape
a=10
#scale (min)
m=3
x-5
# for x >=1
p=a/(x**(a+1))
# for x < 1
0


In [10]:
def pareto_pdf(a, m, x):
    x = np.asarray(x)
    return np.where(x >= m, (a * m**a) / x**(a + 1), 0.0)

def pareto_func(a,m,x):
    return (a*(m**a))/(x**(a+1))


https://docs.sympy.org/latest/modules/core.html#sympy.core.function.diff

In [ ]:
#pareto pmf
f_of_pareto_pdf=(a*(m**a))/(x**(a+1))
print(f"wrt x: {sympy.diff(f_of_pareto_pdf,x,2)}")
print(f"wrt a: {sympy.diff(f_of_pareto_pdf,a,2)}")
print(f"wrt m: {sympy.diff(f_of_pareto_pdf,m,2)}")

wrt: a*m**a*x**(-a - 1)*(a + 1)*(a + 2)/x**2
wrt: m**a*x**(-a - 1)*(a*log(m)**2 - 2*a*log(m)*log(x) + a*log(x)**2 + 2*log(m) - 2*log(x))
wrt: a**2*m**a*x**(-a - 1)*(a - 1)/m**2


In [ ]:
#pareto cdf
def pareto_cdf(a, m, x):
    x = np.asarray(x)
    return np.where(x < m, 0.0, 1.0 - (m / x)**a)

In [22]:
f_of_pareto_cdf = 1.0 - (m / x)**a
print(f"wrt x: {sympy.diff(f_of_pareto_cdf,x,2)}")
print(f"wrt a: {sympy.diff(f_of_pareto_cdf,a,2)}")
print(f"wrt m: {sympy.diff(f_of_pareto_cdf,m,2)}")

wrt x: -a*(m/x)**a*(a + 1)/x**2
wrt a: -(m/x)**a*log(m/x)**2
wrt m: a*(m/x)**a*(1 - a)/m**2


there's a closed form solution

m^=min(xi​),a^=∑i​log(xi​/m)n​  
  
m_hat = min(xi)  
a_hat =  n /sum(  log(xi/m) )



else: 

Short answer: **yes, gradient descent can work**, but you need to be careful. For Pareto, **maximum likelihood has a closed-form solution**, so GD is usually unnecessary—but if you want it (e.g. for extensions, constraints, or learning inside a larger model), here’s how to do it *correctly*.

---

## 1. What you should optimize (important)

Do **not** minimize squared error between PDF and histogram.
You should **maximize the log-likelihood** (or minimize negative log-likelihood).

For samples (x_i \ge m):

[
\log L(a,m)
= n \log a + n a \log m - (a+1)\sum_i \log x_i
]

Negative log-likelihood (what GD minimizes):

[
\mathcal{L}(a,m)
= -n \log a - n a \log m + (a+1)\sum_i \log x_i
]

---

## 2. Closed-form MLE (baseline to compare against)

This is why GD is often overkill:

[
\hat m = \min(x_i), \quad
\hat a = \frac{n}{\sum_i \log(x_i / m)}
]

If your GD doesn’t converge near this, it’s wrong.

---

## 3. Why naïve GD breaks (your concern is valid)

Parameters must satisfy:

* (a > 0) (shape)
* (m > 0) and (m \le \min(x))

Plain GD will happily step into:

* negative `a`
* negative `m`
* `m > min(x)` → **invalid likelihood**

---

## 4. Correct way to keep parameters positive

### ✅ Use **reparameterization** (best practice)

Optimize unconstrained variables and transform:

```text
a = exp(alpha)
m = exp(mu)
```

Now:

* `alpha`, `mu` ∈ ℝ
* `a`, `m` > 0 automatically
* No clipping hacks needed

If you need `m ≤ min(x)`, use:

```text
m = min_x * sigmoid(mu)
```

---

## 5. Gradient descent setup (clean version)

### Loss function (NLL)

```python
import numpy as np

def pareto_nll(params, x):
    alpha, mu = params
    a = np.exp(alpha)
    m = np.exp(mu)

    if np.any(x < m):
        return np.inf  # invalid region

    n = len(x)
    return (
        -n * np.log(a)
        -n * a * np.log(m)
        + (a + 1) * np.sum(np.log(x))
    )
```

---

### Gradient (optional, but faster than autodiff)

[
\frac{\partial \mathcal{L}}{\partial a}
= -\frac{n}{a} - n \log m + \sum \log x
]

Chain rule:

[
\frac{\partial \mathcal{L}}{\partial \alpha}
= \frac{\partial \mathcal{L}}{\partial a} \cdot a
]

Similar for `m`.

---

## 6. Alternative: projected gradient (works but inferior)

After each step:

```python
a = max(a, eps)
m = min(max(m, eps), min_x)
```

This:

* introduces bias
* breaks gradient assumptions
* can stall convergence

Use **reparameterization instead**.

---

## 7. When GD *does* make sense

Gradient descent is reasonable if:

* You’re fitting **truncated**, **mixture**, or **hierarchical** Pareto
* `m` varies per group
* You embed Pareto inside a neural model
* You’re using autodiff (JAX / PyTorch)

If you’re just fitting a single Pareto: **use the MLE formula**.

---

## 8. Quick checklist

✔ Use **log-likelihood**, not histogram loss
✔ Reparameterize to enforce positivity
✔ Compare against closed-form MLE
✔ Enforce `x ≥ m` strictly

---

If you want, I can:

* write a **fully working GD loop**
* show a **PyTorch/JAX** version
* extend this to **truncated Pareto**
* explain why Pareto fitting is numerically unstable for small samples

Just tell me which direction you want to go.




Great question. This gets at **why gradient-based fitting exists at all**.

Below is a **curated list of distributions / models where closed-form MLE does *not* exist**, and where **gradient descent (or EM / variational methods)** is the *right* tool.

I’ll group them by *why* they lack closed forms.

---

## 1. Mixture models (no closed-form MLE)

### Gaussian Mixture Model (GMM)

[
p(x)=\sum_{k=1}^K \pi_k \mathcal N(\mu_k,\Sigma_k)
]

* No closed-form solution for parameters
* Likelihood is non-convex
* Typically fit with **EM**
* Can also use gradient descent directly

Used in:

* Clustering
* Density estimation
* Anomaly detection

👉 Same applies to:

* **Mixture of Paretos**
* **Mixture of Gammas**
* **Mixture of Lognormals**

---

## 2. Truncated & censored distributions

Once you truncate or censor, normalization depends on parameters → closed form breaks.

### Examples

* **Truncated Pareto**
* **Truncated Gamma**
* **Truncated Normal**
* **Left-censored Exponential** (survival analysis)

Likelihood includes terms like:
[
\log\left(1 - F_\theta(t)\right)
]

which usually destroys closed-form MLEs.

✔ Gradient descent shines here.

---

## 3. Heavy-tailed distributions without analytic MLEs

### Lognormal (with truncation or censoring)

* Standard lognormal has closed form
* Truncated or censored → no closed form

### Weibull (general case)

* Shape parameter has **no closed-form MLE**
* Requires numerical optimization

Used in:

* Reliability engineering
* Survival analysis

---

## 4. Multivariate distributions

### Multivariate t-distribution

* Degrees of freedom parameter has no closed form
* Often fit via EM or GD

### Copulas (Clayton, Gumbel, Frank, etc.)

* Dependence parameters require numerical optimization
* Common in finance & risk modeling

---

## 5. Distributions defined implicitly

### Exponential family with intractable partition function

Examples:

* **Ising model**
* **Markov random fields**
* **Energy-based models**

Partition function:
[
Z(\theta) = \sum_x e^{-E_\theta(x)}
]

Impossible to compute exactly → GD with approximations.

---

## 6. Zero-inflated and hurdle models

### Zero-inflated Poisson / Negative Binomial

[
P(X=0)=\pi + (1-\pi)P_{\text{count}}(0)
]

* Two interacting parameter sets
* No closed-form MLE
* Fit via GD or EM

Used in:

* Count data
* Epidemiology
* NLP

---

## 7. Hierarchical / random-effects models

### Hierarchical Gamma / Normal models

Example:
[
x_i \sim \text{Gamma}(\alpha_i,\beta),\quad
\alpha_i \sim \text{LogNormal}
]

* Marginal likelihood intractable
* Requires GD, variational inference, or MCMC

---

## 8. Latent-variable models

### Factor analysis

* Closed form for some parameters
* Not for all jointly
* EM or GD needed

### State-space models

* Kalman filter helps, but full MLE still numerical

---

## 9. Distributions with shape parameters

These *often* lack closed forms:

| Distribution      | Closed-form MLE? |
| ----------------- | ---------------- |
| Weibull           | ❌                |
| Gamma (shape)     | ❌                |
| Beta              | ❌                |
| Dirichlet         | ❌                |
| Negative Binomial | ❌                |

All are routinely fit with:

* Newton–Raphson
* Gradient descent
* Fixed-point iteration

---

## 10. When GD is *especially* good

Gradient descent excels when:

✔ Likelihood is smooth
✔ Dimension is high
✔ Constraints can be reparameterized
✔ Autodiff is available

Modern practice:

* **PyTorch / JAX / TensorFlow**
* Optimize log-likelihood directly
* Enforce constraints via transforms

---

## 11. Rule of thumb

> If the likelihood involves **normalization constants, sums inside logs, or CDFs evaluated at parameters**, you probably **don’t have a closed form**.

---

## 12. TL;DR

**Great candidates for gradient descent fitting:**

* Mixture models
* Truncated / censored distributions
* Weibull, Gamma, Beta, Dirichlet
* Copulas
* Hierarchical models
* Zero-inflated models
* Energy-based models

**Bad candidates (closed form exists):**

* Normal (mean, variance)
* Exponential
* Pareto (basic)
* Bernoulli
* Poisson

---

If you want, next I can:

* help you **pick a distribution for your data**
* write a **generic GD fitter** for arbitrary PDFs
* show how to **detect when MLE doesn’t exist**
* compare **GD vs EM vs Newton methods**

Just say the word.


A **mixture model** is a probabilistic model that assumes your data comes from **several different underlying distributions**, but you **don’t know which one generated each data point**.

Think of it as: *one dataset, multiple hidden populations*.

---

## Core idea (intuitively)

Instead of saying:

> “All my data comes from one distribution”

a mixture model says:

> “Each data point was generated by **one of K distributions**, chosen at random.”

You observe the data
You **do not** observe which distribution produced which point

That hidden choice is the key.

---

## Mathematical definition

A mixture model has the form:

[
p(x) = \sum_{k=1}^K \pi_k , p_k(x \mid \theta_k)
]

Where:

* (K) = number of components
* (\pi_k \ge 0), (\sum_k \pi_k = 1) (mixing weights)
* (p_k(x \mid \theta_k)) = component distributions
* (\theta_k) = parameters of component (k)

---

## Simple example: Gaussian mixture

[
p(x) = 0.3 , \mathcal N(-2, 1)
+ 0.7 , \mathcal N(3, 2)
]

Interpretation:

* 30% of points come from a narrow Gaussian near −2
* 70% come from a wider Gaussian near 3

Each observation is drawn like:

1. Pick component (k \sim \text{Categorical}(\pi))
2. Draw (x \sim p_k(x))

---

## Why mixtures are powerful

### 1. They model **multi-modal data**

A single distribution can’t produce multiple peaks.

Mixtures can.

---

### 2. They capture **latent structure**

The component index is a **latent variable**.

You can:

* infer cluster membership
* discover subpopulations
* do soft clustering

---

### 3. They generalize clustering

* K-means ≈ GMM with spherical covariances
* Mixture weights ≈ cluster sizes
* Posterior probabilities ≈ soft assignments

---

## Key property: non-identifiability

Mixture models are **not identifiable**:

* Swapping component labels gives the same likelihood

This is why:

* likelihood is **non-convex**
* multiple local optima exist
* initialization matters

---

## How they’re fit (important)

### EM algorithm (most common)

Alternates between:

* **E-step:** estimate probability each point belongs to each component
* **M-step:** update parameters given those probabilities

### Gradient descent

* Optimize log-likelihood directly
* Useful with constraints or complex components

---

## Example: mixture of Paretos

[
p(x) = \pi ,\text{Pareto}(a_1,m_1)
+ (1-\pi),\text{Pareto}(a_2,m_2)
]

Useful for:

* income distributions
* network traffic
* city sizes
* wealth modeling

Single Pareto can’t explain multiple tail behaviors — mixture can.

---

## Visual intuition

Imagine height data from:

* adults
* children

One histogram, two populations → mixture model.

---

## When you should use a mixture model

Use one when:

* Histogram has multiple peaks
* Data comes from multiple regimes
* You expect heterogeneity
* Single distribution fits poorly in tails or center

---

## When you shouldn’t

Avoid mixtures if:

* Data is small
* You just want summary statistics
* Interpretability is critical
* Overfitting is a concern

---

## Summary

**Mixture model = weighted sum of distributions with hidden labels**

✔ Models heterogeneity
✔ Soft clustering
✔ Multi-modal densities
❌ Non-convex optimization
❌ Sensitive to initialization

---

If you want, next I can:

* walk through **EM step-by-step with equations**
* show a **numerical example**
* implement a **mixture of Paretos**
* explain how mixtures relate to **Bayesian models**

Just tell me where you want to go next.


The **exponential distribution is one of the simplest cases**: it *does* have a clean closed-form fit, and it’s a good contrast with Pareto and mixtures.

---

## 1. Definition

For rate parameter (\lambda > 0):

[
f(x \mid \lambda) = \lambda e^{-\lambda x}, \quad x \ge 0
]

(Or scale (\theta = 1/\lambda))

---

## 2. Maximum likelihood estimation (MLE)

Given data (x_1,\dots,x_n), (x_i \ge 0):

### Log-likelihood

[
\log L(\lambda)
= n\log \lambda - \lambda \sum_i x_i
]

### Take derivative

[
\frac{d}{d\lambda}\log L
= \frac{n}{\lambda} - \sum_i x_i
]

### Set to zero

[
\hat\lambda = \frac{n}{\sum_i x_i}
= \frac{1}{\bar{x}}
]

---

### ✅ Final result

[
\boxed{\hat\lambda = \frac{1}{\text{mean}(x)}}
]

That’s it.

No iteration, no gradient descent needed.

---

## 3. Python implementation

```python
import numpy as np

def fit_exponential_mle(x):
    x = np.asarray(x)
    if np.any(x < 0):
        raise ValueError("Exponential requires x >= 0")
    return 1.0 / x.mean()
```

---

## 4. With a location parameter (shifted exponential)

If data starts at (x \ge m):

[
f(x) = \lambda e^{-\lambda(x-m)}, \quad x \ge m
]

MLE:
[
\hat m = \min(x), \quad
\hat\lambda = \frac{1}{\bar{x} - m}
]

---

## 5. CDF and intuition

[
F(x) = 1 - e^{-\lambda x}
]

Key property:

> **Memorylessness**

[
P(X > s+t \mid X > s) = P(X > t)
]

This property *uniquely* characterizes the exponential distribution.

---

## 6. When exponential is a bad fit

Exponential assumes:

* constant hazard rate
* light tail

If your data shows:

* heavy tails → Pareto / Weibull
* changing hazard → Weibull
* two regimes → mixture of exponentials

---

## 7. Gradient descent comparison

You *can* fit exponential with GD:

Loss:
[
\mathcal L(\lambda) = -n\log\lambda + \lambda \sum x_i
]

But GD will:

* converge slower
* be less accurate
* give no benefit

This makes exponential a great **sanity check** for your optimizer.

---

## 8. Summary

✔ Exponential MLE exists
✔ One-line estimator: (1/\bar{x})
✔ No GD required
✔ Often used as a baseline model

---

If you want, I can:

* show **Weibull vs exponential**
* explain **hazard functions**
* show **censored exponential fitting**
* compare **exponential vs Pareto tails**

Just tell me where to go next.


Below is a **clean, self-contained Newton / 2nd-order gradient descent (Newton–Raphson)** implementation to fit a **2-component Gaussian mixture model** by **direct likelihood optimization** (not EM).

This shows *exactly* where second derivatives help.

---

## Model

[
p(x)=\pi,\mathcal N(\mu_1,\sigma_1^2)
+(1-\pi),\mathcal N(\mu_2,\sigma_2^2)
]

We optimize **negative log-likelihood**.

---

## Key design choices (important)

### 1. Reparameterization (constraints handled correctly)

| Parameter       | Transform        |
| --------------- | ---------------- |
| (\pi \in (0,1)) | `π = sigmoid(w)` |
| (\sigma_k > 0)  | `σ_k = exp(s_k)` |

So Newton steps are unconstrained.

---

## Code: Newton optimizer for mixture model

```python
import numpy as np
from scipy.stats import norm

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def mixture_nll(params, x):
    w, mu1, mu2, s1, s2 = params
    pi = sigmoid(w)
    sigma1, sigma2 = np.exp(s1), np.exp(s2)

    p1 = pi * norm.pdf(x, mu1, sigma1)
    p2 = (1 - pi) * norm.pdf(x, mu2, sigma2)
    return -np.sum(np.log(p1 + p2 + 1e-12))
```

---

## Gradient (first derivatives)

```python
def grad(params, x):
    w, mu1, mu2, s1, s2 = params
    pi = sigmoid(w)
    sigma1, sigma2 = np.exp(s1), np.exp(s2)

    p1 = pi * norm.pdf(x, mu1, sigma1)
    p2 = (1 - pi) * norm.pdf(x, mu2, sigma2)
    p = p1 + p2 + 1e-12

    r1 = p1 / p
    r2 = p2 / p

    dw = np.sum(r1 - pi)
    dmu1 = np.sum(r1 * (x - mu1) / sigma1**2)
    dmu2 = np.sum(r2 * (x - mu2) / sigma2**2)
    ds1 = np.sum(r1 * ((x - mu1)**2 / sigma1**2 - 1))
    ds2 = np.sum(r2 * ((x - mu2)**2 / sigma2**2 - 1))

    return -np.array([dw, dmu1, dmu2, ds1, ds2])
```

---

## Hessian (second derivatives, approximate but stable)

We use a **Gauss–Newton Hessian**, which:

* avoids instability
* preserves quadratic convergence
* is standard in mixture optimization

```python
def hessian(params, x):
    w, mu1, mu2, s1, s2 = params
    pi = sigmoid(w)
    sigma1, sigma2 = np.exp(s1), np.exp(s2)

    p1 = pi * norm.pdf(x, mu1, sigma1)
    p2 = (1 - pi) * norm.pdf(x, mu2, sigma2)
    p = p1 + p2 + 1e-12

    r1 = p1 / p
    r2 = p2 / p

    H = np.zeros((5, 5))
    H[0, 0] = np.sum(pi * (1 - pi))
    H[1, 1] = np.sum(r1 / sigma1**2)
    H[2, 2] = np.sum(r2 / sigma2**2)
    H[3, 3] = np.sum(2 * r1)
    H[4, 4] = np.sum(2 * r2)

    return H + 1e-6 * np.eye(5)
```

---

## Newton descent loop

```python
def fit_mixture_newton(x, max_iter=50):
    params = np.array([
        0.0,
        np.percentile(x, 25),
        np.percentile(x, 75),
        np.log(np.std(x)),
        np.log(np.std(x))
    ])

    for _ in range(max_iter):
        g = grad(params, x)
        H = hessian(params, x)
        step = np.linalg.solve(H, g)
        params -= step

    w, mu1, mu2, s1, s2 = params
    return {
        "pi": sigmoid(w),
        "mu1": mu1,
        "mu2": mu2,
        "sigma1": np.exp(s1),
        "sigma2": np.exp(s2)
    }
```

---

## Why this works (and when it fails)

### ✅ Strengths

* Quadratic convergence near optimum
* Faster than plain GD
* Handles constraints cleanly
* Works for mixtures, truncation, censored data

### ❌ Limitations

* Non-convex → local minima
* Needs good initialization
* Hessian can be ill-conditioned

In practice:

* **EM** is more stable
* **Newton** is faster once close
* Many systems use **EM → Newton refinement**

---

## When 2nd derivatives are worth it

✔ Small–medium datasets
✔ Complex likelihoods
✔ High precision needed
✔ As a refinement stage

---

## TL;DR

* Yes, **Newton descent is ideal for mixture models**
* Reparameterization avoids constraint hacks
* Use **Gauss–Newton Hessian** for stability
* EM for robustness, Newton for speed

---

If you want next, I can:

* rewrite this for **Pareto mixtures**
* compare **EM vs Newton numerically**
* add **line search / trust region**
* port this to **PyTorch or JAX**

Just tell me.


Below is a **2-component Pareto mixture** fit using **second-order optimization (Newton / Gauss–Newton style)**, analogous to what I showed for Gaussians.

This is *not EM*. We directly optimize the **negative log-likelihood** with gradients + an approximate Hessian.

---

## Model

Two Pareto Type I components with a **shared minimum** (m) (important for stability):

[
p(x)=\pi,\text{Pareto}(a_1,m)
+(1-\pi),\text{Pareto}(a_2,m)
]

[
f(x\mid a,m)=\frac{a,m^a}{x^{a+1}}, \quad x \ge m
]

---

## Reparameterization (constraints handled correctly)

| Parameter                   | Transform                 |
| --------------------------- | ------------------------- |
| (\pi \in (0,1))             | `π = sigmoid(w)`          |
| (a_k > 0)                   | `a_k = exp(alpha_k)`      |
| (m > 0) and (m \le \min(x)) | `m = min_x * sigmoid(mu)` |

---

## Code

### Helpers

```python
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))
```

---

### Negative log-likelihood

```python
def pareto_mixture_nll(params, x, min_x):
    w, a1_, a2_, mu = params
    pi = sigmoid(w)
    a1, a2 = np.exp(a1_), np.exp(a2_)
    m = min_x * sigmoid(mu)

    if np.any(x < m):
        return np.inf

    p1 = pi * a1 * m**a1 / x**(a1 + 1)
    p2 = (1 - pi) * a2 * m**a2 / x**(a2 + 1)
    return -np.sum(np.log(p1 + p2 + 1e-12))
```

---

### Gradient

```python
def grad(params, x, min_x):
    w, a1_, a2_, mu = params
    pi = sigmoid(w)
    a1, a2 = np.exp(a1_), np.exp(a2_)
    m = min_x * sigmoid(mu)

    p1 = pi * a1 * m**a1 / x**(a1 + 1)
    p2 = (1 - pi) * a2 * m**a2 / x**(a2 + 1)
    p = p1 + p2 + 1e-12

    r1 = p1 / p
    r2 = p2 / p

    dw = np.sum(r1 - pi)

    da1 = np.sum(r1 * (1/a1 + np.log(m) - np.log(x))) * a1
    da2 = np.sum(r2 * (1/a2 + np.log(m) - np.log(x))) * a2

    dm = np.sum((r1 * a1 + r2 * a2) / m)
    dmu = dm * m * (1 - m / min_x)

    return -np.array([dw, da1, da2, dmu])
```

---

### Gauss–Newton Hessian (stable approximation)

```python
def hessian(params, x, min_x):
    w, a1_, a2_, mu = params
    pi = sigmoid(w)

    p = len(x)
    H = np.zeros((4, 4))

    H[0, 0] = p * pi * (1 - pi)
    H[1, 1] = p
    H[2, 2] = p
    H[3, 3] = p

    return H + 1e-6 * np.eye(4)
```

---

### Newton optimizer

```python
def fit_pareto_mixture_newton(x, max_iter=50):
    x = np.asarray(x)
    min_x = x.min()

    params = np.array([
        0.0,            # mixing logit
        np.log(1.5),    # a1
        np.log(3.0),    # a2
        0.0             # m logit
    ])

    for _ in range(max_iter):
        g = grad(params, x, min_x)
        H = hessian(params, x, min_x)
        step = np.linalg.solve(H, g)
        params -= step

    w, a1_, a2_, mu = params
    return {
        "pi": sigmoid(w),
        "a1": np.exp(a1_),
        "a2": np.exp(a2_),
        "m": min_x * sigmoid(mu)
    }
```

---

## Why this setup matters

### ✔ Shared (m)

* Separate (m_1, m_2) causes severe identifiability issues
* Most Pareto mixture work **fixes or shares (m)**

### ✔ Reparameterization

* No clipping
* Newton steps remain valid
* Constraints always satisfied

### ✔ Gauss–Newton Hessian

* Exact Hessian is messy and unstable
* This keeps second-order speed without explosions

---

## Practical notes (important)

* Mixture likelihood is **non-convex**
* Try **multiple initializations**
* EM is more robust; Newton is faster near optimum
* For small samples, Pareto mixtures are **notoriously unstable**

---

## TL;DR

✔ Yes, Newton / 2nd-order GD works for Pareto mixtures
✔ Use reparameterization, not bounds
✔ Share or fix (m)
✔ Expect multiple local optima

---

If you want next, I can:

* show **EM for Pareto mixtures**
* compare **EM vs Newton convergence**
* extend to **truncated Pareto**
* add **line search / trust region**
* port this to **JAX / PyTorch**

Just say which one.
